# Business Vault: Bridge Tables

## Setup

### Imports

In [1]:
from pathlib import Path

import duckdb

### Configuration

In [2]:
# Locate the project root (folder containing requirements.txt), independent of cwd
PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists()
)

data_vault_path = str(PROJECT_ROOT / "data" / "warehouse" / "data_vault.duckdb")

### DuckDB Connection

In [3]:
con = duckdb.connect(data_vault_path)

In [4]:
con.execute("""
CREATE SCHEMA IF NOT EXISTS business_vault;
""")

### Order Bridge

In [5]:
con.execute("""
CREATE OR REPLACE TABLE business_vault.bridge_order AS
SELECT
    o.order_hk,
    oc.customer_hk

FROM raw_vault.hub_order o

LEFT JOIN raw_vault.link_order_customer oc
    ON o.order_hk = oc.order_hk;
""")

### Order Payment Bridge

In [6]:
con.execute("""
CREATE OR REPLACE TABLE business_vault.bridge_order_payment AS
SELECT
    op.order_payment_hk,
    op.order_hk,
    op.payment_sequential,
    oc.customer_hk

FROM raw_vault.link_order_payment op

LEFT JOIN raw_vault.link_order_customer oc
    ON op.order_hk = oc.order_hk;
""")

### Order Review Bridge

In [7]:
con.execute("""
CREATE OR REPLACE TABLE business_vault.bridge_order_review AS
SELECT
    r.order_review_hk,
    r.order_hk,
    r.review_hk,
    oc.customer_hk

FROM raw_vault.link_order_review r

LEFT JOIN raw_vault.link_order_customer oc
    ON r.order_hk = oc.order_hk;
""")

### Order Item Bridge

In [8]:
con.execute("""
CREATE OR REPLACE TABLE business_vault.bridge_order_item AS
SELECT
    oi.order_item_hk,
    oi.order_item_id,

    oi.order_hk,
    oc.customer_hk,
    oi.product_hk,
    oi.seller_hk

FROM raw_vault.link_order_item oi

LEFT JOIN raw_vault.link_order_customer oc
    ON oi.order_hk = oc.order_hk;
""")